In [2]:
import numpy
import sympy as sp

del_p, rho, g, del_h = sp.symbols('del_p rho g del_h', real=True, positive=True)
d_A, L_A, d_B, L_B, visc = sp.symbols('d_A L_A d_B L_B visc', real=True, positive=True)
f_A, f_B = sp.symbols('f_A f_B', real=True, positive=True)
v_b = sp.Symbol('v_b', real=True)


eq_main = sp.Eq(del_p/(rho*g),
            del_h + (v_b**2)/(2*g) +
            0.5*((1-(d_B**2/d_A**2))**2 + 2*0.08 + f_A*L_A/d_A)*(3*d_B**2/d_A**2)*(v_b**2) +
            0.5*(2*0.008 + f_B*L_B/d_B + 2*0.3)*v_b**2)
eq_fB = sp.Eq(
    1/sp.sqrt(f_B),
    2*sp.log((v_b * d_B * sp.sqrt(f_B)) / (2.51 * visc), 10 )
)

eq_fA = sp.Eq(
    1/sp.sqrt(f_A),
    2*sp.log( ((3*v_b*(d_B**2/d_A**2)) * d_A * sp.sqrt(f_A)) / (2.51 * visc), 10 )
)

param_values = {
    del_p: 335000-30000, 
    rho:   1000,    
    g:     9,   
    del_h: (82+100+3-14)/1000,     # Example: 10 m
    L_A:   82/1000,    # Example pipe length
    d_A:   20/1000,   # Example pipe diameter
    L_B:   (100+3+14)/1000,
    d_B:   12/1000,
    visc: 1e-6
}

eq1_sub = eq_main.subs(param_values)
eq2_sub = eq_fB.subs(param_values)
eq3_sub = eq_fA.subs(param_values)

# 5. Solve the system numerically.
#    nonlinsolve() attempts a symbolic or numeric approach to the system of equations.
#    You might get a symbolic solution, a numeric solution, or an empty/conditional set.
solution_set = sp.nonlinsolve([eq1_sub, eq2_sub, eq3_sub], [v_b, f_A, f_B])

print("Candidate solution(s) from nonlinsolve:")
print(solution_set)

# If nonlinsolve returns a solution set, you can attempt .evalf() on each solution:
for sol in solution_set:
    print("Evaluated solution:")
    print([expr.evalf() for expr in sol])

Candidate solution(s) from nonlinsolve:
EmptySet


In [13]:
import sympy as sp

# ---------------------------
# 1) Define symbols and equations
# ---------------------------

del_p, rho, g, del_h = sp.symbols('del_p rho g del_h', real=True, positive=True)
d_A, L_A, d_B, L_B, visc = sp.symbols('d_A L_A d_B L_B visc', real=True, positive=True)
f_A, f_B = sp.symbols('f_A f_B', real=True, positive=True)
v_b = sp.Symbol('v_b', real=True, positive=True)

# Main pressure/energy balance equation
eq_main = sp.Eq(
    del_p/(rho*g),
    del_h
    + (v_b**2)/(2*g)
    + 0.5 * (
        (1 - (d_B**2/d_A**2))**2
        + 2*0.08
        + f_A*L_A/d_A
    )*(3*d_B**2/d_A**2)*(v_b**2)
    + 0.5 * (
        2*0.008
        + f_B*L_B/d_B
        + 2*0.3
    ) * (v_b**2)
)

# Colebrook-like equation for f_B
eq_fB = sp.Eq(
    1/sp.sqrt(f_B),
    2 * sp.log((v_b * d_B * sp.sqrt(f_B)) / (2.51 * visc), 10)
)

# Colebrook-like equation for f_A
eq_fA = sp.Eq(
    1/sp.sqrt(f_A),
    2 * sp.log(((3*v_b*(d_B**2/d_A**2)) * d_A * sp.sqrt(f_A)) / (2.51 * visc), 10)
)

# ---------------------------
# 2) Numeric parameter values
#    (Replace with your actual known data)
# ---------------------------
param_values = {
    del_p: 335000 - 30000,  # example: 305000 Pa
    rho:   1000,           # 1000 kg/m^3
    g:     9,              # 9 m/s^2
    del_h: (82 + 100 + 3 - 14) / 1000,  # Just as you had, in meters
    L_A:   82/1000,
    d_A:   20/1000,
    L_B:   (100 + 3 + 14)/1000,
    d_B:   12/1000,
    visc:  1e-6            # e.g. 1e-6 m^2/s (kinematic viscosity)
}

# ---------------------------
# 3) Partially substitute known parameters into each equation
#    so that only v_b, f_A, f_B remain unknown
# ---------------------------
eq_main_sub = eq_main.xreplace(param_values)
eq_fB_sub   = eq_fB.xreplace(param_values)
eq_fA_sub   = eq_fA.xreplace(param_values)


# ---------------------------
# 4) Routines that solve for one variable at a time using nsolve
#    with a list of guesses until success
# ---------------------------

def solve_v_b(current_fA, current_fB, guess_list=[0.001, 0.01, 0.5, 1, 2, 5]):
    """
    Solve eq_main_sub for v_b, given fixed f_A & f_B, trying multiple guesses.
    """
    eq_tmp = eq_main_sub.subs({f_A: current_fA, f_B: current_fB})
    for guess in guess_list:
        try:
            sol = sp.nsolve(eq_tmp.lhs - eq_tmp.rhs, v_b, guess)
            # Domain check: v_b must keep logs positive in eq_fB, eq_fA (non-zero)
            if sol > 0:
                return float(sol)
        except:
            continue
    raise ValueError("Cannot solve eq_main_sub for v_b with given f_A, f_B and guess_list.")

def solve_f_B(current_v_b, guess_list=[0.001, 0.01, 0.02, 0.03]):
    """
    Solve eq_fB_sub for f_B, given v_b, trying multiple friction factor guesses.
    """
    eq_tmp = eq_fB_sub.subs({v_b: current_v_b})
    for guess in guess_list:
        try:
            sol = sp.nsolve(eq_tmp.lhs - eq_tmp.rhs, f_B, guess)
            # f_B must be > 0 and also keep log(...) argument in eq_fB positive
            if sol > 0:
                # Check log argument:
                arg = (current_v_b * param_values[d_B] * sol**0.5)/(2.51*param_values[visc])
                if arg > 0:
                    return float(sol)
        except:
            continue
    raise ValueError("Cannot solve eq_fB_sub for f_B with given v_b and guess_list.")

def solve_f_A(current_v_b, guess_list=[0.001, 0.01, 0.02, 0.03]):
    eq_tmp = eq_fA_sub.subs({v_b: current_v_b})
    for guess in guess_list:
        try:
            print(f"  Trying f_A guess = {guess} for v_b = {current_v_b}...")
            sol = sp.nsolve(eq_tmp.lhs - eq_tmp.rhs, f_A, guess)
            # Check positivity
            if sol <= 0:
                print(f"    Found negative f_A = {sol}. Skipping.")
                continue
            # Check log argument
            arg = (3*current_v_b*(param_values[d_B]**2/param_values[d_A]**2)
                   * param_values[d_A] * sol**0.5)/(2.51*param_values[visc])
            if arg <= 0:
                print(f"    Log arg <= 0 => {arg}. Skipping.")
                continue
            return float(sol)
        except Exception as e:
            print(f"    nsolve failed for guess {guess}. Error: {e}")
            continue
    raise ValueError("Cannot solve eq_fA_sub for f_A with given v_b and guess_list.")

# ---------------------------
# 5) The iterative loop
# ---------------------------
def iterative_solve(
    max_iterations=100,
    tolerance=1e-2,
    v_b_init=1.0,
    fA_init=0.02,
    fB_init=0.02
):
    """
    Iteratively update (v_b, f_A, f_B) until convergence or max_iterations.
    """
    v_b_val = v_b_init
    fA_val  = fA_init
    fB_val  = fB_init
    
    for i in range(max_iterations):
        old_v_b, old_fA, old_fB = v_b_val, fA_val, fB_val
        
        # 1) Solve eq_main_sub for v_b
        v_b_val = solve_v_b(fA_val, fB_val, guess_list=[old_v_b, 0.1, 0.5, 1, 2, 5, 6, 7])
        
        # 2) Solve eq_fB_sub for f_B
        fB_val = solve_f_B(v_b_val, guess_list=[old_fB, 0.008, 0.01, 0.02, 0.03])
        
        # 3) Solve eq_fA_sub for f_A
        fA_val = solve_f_A(v_b_val, guess_list=[old_fA, 0.008, 0.01, 0.02, 0.03])
        
        # Check convergence
        diff_v_b = abs(v_b_val - old_v_b)
        diff_fA  = abs(fA_val - old_fA)
        diff_fB  = abs(fB_val - old_fB)
        
        if diff_v_b < tolerance and diff_fA < tolerance and diff_fB < tolerance:
            print(f"Converged in {i+1} iteration(s).")
            break
    
    return v_b_val, fA_val, fB_val

v_b_final, fA_final, fB_final = iterative_solve()
print("\nFinal results after iteration:")
print(f"v_b = {v_b_final:.6f} m/s")
print(f"f_A = {fA_final:.6f}")
print(f"f_B = {fB_final:.6f}")

  Trying f_A guess = 0.02 for v_b = 6.440304509191741...
    nsolve failed for guess 0.02. Error: Could not find root within given tolerance. (1122.51089944226407476 > 2.16840434497100886801e-19)
Try another starting point or tweak arguments.
  Trying f_A guess = 0.008 for v_b = 6.440304509191741...
    nsolve failed for guess 0.008. Error: Could not find root within given tolerance. (1202.56231019789456378 > 2.16840434497100886801e-19)
Try another starting point or tweak arguments.
  Trying f_A guess = 0.01 for v_b = 6.440304509191741...
    nsolve failed for guess 0.01. Error: Could not find root within given tolerance. (1163.46023532628717409 > 2.16840434497100886801e-19)
Try another starting point or tweak arguments.
  Trying f_A guess = 0.02 for v_b = 6.440304509191741...
    nsolve failed for guess 0.02. Error: Could not find root within given tolerance. (1122.51089944226407476 > 2.16840434497100886801e-19)
Try another starting point or tweak arguments.
  Trying f_A guess = 0.03 

ValueError: Cannot solve eq_fA_sub for f_A with given v_b and guess_list.

In [25]:
import math

def friction_factor_haaland(Re, roughness, diameter):
    """
    Returns the friction factor f using Haaland's Equation:
      1 / sqrt(f) = -1.8 * log10( ( (k/D)/3.7 )^1.11 + 6.9/Re )
    where k is the pipe roughness height.

    For laminar flow (Re < ~2000), we switch to 64/Re as an approximation.
    """
    if Re < 1e-6:
        # Guard against a nearly zero Reynolds number
        return 0.0
    if Re < 2000:
        # Laminar regime
        return 64.0 / Re

    term = ((roughness/diameter)/3.7)**1.11 + (6.9/Re)
    return 1.0 / ( -1.8 * math.log10(term) )**2


def main_equation_residual(v_b, param):
    """
    Computes the residual of the main pressure/energy equation:

      residual(v_b) = (del_p/(rho*g)) - [ del_h
                                          + v_b^2/(2*g)
                                          + 0.5*( ... f_A-based terms ... )*v_b^2
                                          + 0.5*( ... f_B-based terms ... )*v_b^2 ]

    where f_A, f_B are computed from a direct friction correlation.
    """
    # Unpack parameters for readability
    dp   = param['del_p']
    rho  = param['rho']
    g    = param['g']
    delh = param['del_h']
    dA   = param['d_A']
    dB   = param['d_B']
    LA   = param['L_A']
    LB   = param['L_B']
    nu   = param['visc']         # kinematic viscosity
    epsA = param['eps_A']        # roughness pipe A
    epsB = param['eps_B']        # roughness pipe B

    # Reynolds numbers for each pipe
    ReA = (v_b * dA) / nu
    ReB = (v_b * dB) / nu

    # Compute friction factors via Haaland (or any direct correlation)
    fA = friction_factor_haaland(ReA, epsA, dA)
    fB = friction_factor_haaland(ReB, epsB, dB)

    # Right-hand side of your main equation
    # (matches the eq_main structure you provided)
    rhs = ( delh
            + (v_b**2)/(2*g)
            + 0.5 * (
                (1 - (dB**2/dA**2))**2
                + 2*0.08
                + fA*LA/dA
            )*(3*dB**2/dA**2)*(v_b**2)
            + 0.5 * (
                2*0.008
                + fB*LB/dB
                + 2*0.3
            )*(v_b**2)
          )

    lhs = dp/(rho*g)   # left-hand side

    return lhs - rhs   # residual = LHS - RHS


def solve_velocity_bisection(param, v_b_low=1e-5, v_b_high=5.0, tol=1e-6, max_iter=100):
    """
    Solve for v_b by bracket-bisection:
      1) Evaluate residual at v_b_low and v_b_high.
      2) Keep halving the bracket until residual is close to zero
         or we reach max_iter.

    Constraints:
      - main_equation_residual(v_b_low) and main_equation_residual(v_b_high)
        must have opposite signs (one positive, one negative).
    """
    f_low = main_equation_residual(v_b_low, param)
    f_high = main_equation_residual(v_b_high, param)

    if f_low * f_high > 0:
        raise ValueError(
            "Bisection method: The signs of residual at v_b_low and v_b_high must be different.\n"
            f"f_low={f_low}, f_high={f_high}"
        )

    for iteration in range(max_iter):
        v_b_mid = 0.5*(v_b_low + v_b_high)
        f_mid = main_equation_residual(v_b_mid, param)

        if abs(f_mid) < tol:
            # Converged
            return v_b_mid

        # Narrow down bracket based on the sign of the residual
        if (f_low * f_mid) < 0:
            # Root is between v_b_low and v_b_mid
            v_b_high = v_b_mid
            f_high = f_mid
        else:
            # Root is between v_b_mid and v_b_high
            v_b_low = v_b_mid
            f_low = f_mid

    # If we exit the loop, no convergence within max_iter
    raise ValueError("Bisection did not converge within max_iter iterations.")

param_values = {
    'del_p' : 335000 - 30000,  # Pa
    'rho'   : 1000.0,         # kg/m^3
    'g'     : (9.78+9.55)/2,            # m/s^2
    'del_h' : 0.175,  # m
    'd_A'   : 20.0e-3,        # m
    'L_A'   : 82.0e-3,        # m
    'd_B'   : 12.0e-3,        # m
    'L_B'   : 0.117,  # m
    'visc'  : 1.0e-6,         # m^2/s (kinematic viscosity)
    'eps_A' : 1e-6,         # Pipe roughness (m) for A
    'eps_B' : 1e-6,         # Pipe roughness (m) for B
}

# 1) Choose a bracket for velocity in [0.00001, 5.0] m/s
# 2) Solve using bisection
v_b_solution = solve_velocity_bisection(param_values, v_b_low=1e-6, v_b_high=20.0)

# Print the final velocity
print(f"Final velocity solution = {v_b_solution:.6f} m/s")
print(param_values['rho']*v_b_solution*math.pi*(param_values['d_B']/2)**2)
print((v_b_solution * param_values['d_B']) / param_values['visc'])


Final velocity solution = 6.265960 m/s
0.708663327639099
75191.51438780934
